# Full-Catalog Two-Stage Eval (First-Token Filter + Full LL)

**Status:** future work — not run as part of the original submission.

## What this does

Approximate full-catalog log-likelihood scoring with two stages per user:

1. **Stage 1 — first-token prefilter.** Forward-pass the prompt once. From the resulting
   next-token distribution, score every catalog item by the log-probability of its
   first token. Keep the top-N candidates (default N=500). This is essentially free —
   one prompt pass + a vectorized lookup over 6,117 token IDs.
2. **Stage 2 — full log-likelihood on survivors.** For the top-N from stage 1, compute
   the full per-token mean log-probability using the cached prompt KV. Re-rank.

Items not in the top-N from stage 1 get `-inf` and are placed at the bottom of the final
ranking.

## Why this exists

The exact full-catalog notebook (`01_full_catalog_kvcache.ipynb`) takes ~3–6 days
even with KV caching. This two-stage version converts the problem to roughly the same
cost as the shared-pool eval — about 30–75 hours total — at the cost of approximation.

## Accuracy caveat

First-token prefiltering can miss titles whose first token is unlikely under the model
but whose full sequence has high log-likelihood. In practice, a top-500 stage-1 cut
captures the great majority of items that would rank in the final top-20, but this
is an empirical claim that should be checked by running both notebooks on a small
subset of users and comparing the resulting top-20 lists.

## Requirements

- Colab A100 or L4
- QLoRA adapter from `03_1_genrec_qlora_finetuning.ipynb`
- `SMOKE_TEST = True` runs on 3 users to verify the pipeline before committing

## 0. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets torch

In [ ]:
SMOKE_TEST = True       # 3 users, no save — set False for the real run
TOP_N_STAGE1 = 500      # how many items survive the first-token filter
TOP_K_TO_SAVE = 50      # how many items to store per user in the final ranking

if SMOKE_TEST:
    N_USERS = 3
    CHECKPOINT_EVERY = 1
    print('*** SMOKE TEST: 3 users, no save ***')
else:
    N_USERS = None
    CHECKPOINT_EVERY = 100
    print(f'*** FULL RUN — stage1 keeps top-{TOP_N_STAGE1}, saving top-{TOP_K_TO_SAVE} per user ***')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/Foundations of Large Language Models/Final Project'
DATA_DIR = f'{DRIVE_ROOT}/data'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
ADAPTER_DIR = f'{DRIVE_ROOT}/results/genrec_qlora/final_adapter'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), \
    f'No adapter found at {ADAPTER_DIR}. Run notebook 03_1 first.'
print(f'Adapter found at {ADAPTER_DIR}')

## 1. Load data and model

In [ ]:
import json
import pickle
import numpy as np

with open(f'{DATA_DIR}/shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
with open(f'{DATA_DIR}/genrec_test.json') as f:
    genrec_test = json.load(f)

item_titles = shared['item_titles']
test_ground_truth = shared['test_ground_truth']

print(f'Test users: {len(test_ground_truth)}')
print(f'Catalog items: {len(item_titles)}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print(f'Model loaded with adapter. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Pre-compute catalog title first-token IDs (once)

Stage 1 scores items by their first token's log-probability under the prompt's
next-token distribution. Build a tensor `first_tokens[i] = tokenizer(title_i).input_ids[0]`
now so the per-user stage 1 step is just one tensor indexing operation.

In [ ]:
catalog_item_ids = sorted(item_titles.keys())
catalog_titles = [item_titles[i] for i in catalog_item_ids]

first_token_list = []
for title in catalog_titles:
    if not title:
        first_token_list.append(0)  # sentinel; will be masked below
        continue
    ids = tokenizer(title, add_special_tokens=False)['input_ids']
    first_token_list.append(ids[0] if ids else 0)

first_tokens = torch.tensor(first_token_list, dtype=torch.long).to(model.device)
valid_mask = torch.tensor(
    [bool(t) for t in catalog_titles], dtype=torch.bool
).to(model.device)

print(f'Pre-computed first tokens for {len(first_tokens)} catalog items')
print(f'Valid (non-empty) titles: {int(valid_mask.sum())}')

## 3. Two-stage scoring function

In [ ]:
import torch.nn.functional as F


def two_stage_score(prompt_text, top_n_stage1=TOP_N_STAGE1):
    """Two-stage scoring over the full catalog.

    Returns a numpy array of length len(catalog_titles); items not in stage-1 top-N
    receive -inf. The prompt KV is computed once and reused across all stage-2 calls.

    Implementation notes:
    - Stage 2 tokenizes (prompt + title) together and slices off the prompt portion
      to match the naive reference scorer exactly.
    - Stage 2 passes explicit `position_ids` so title tokens are treated as positions
      prompt_len, prompt_len+1, ... rather than 0, 1, ... (without this, scores
      diverge significantly from the naive reference).
    """
    prompt_inputs = tokenizer(prompt_text, return_tensors='pt')
    prompt_ids = prompt_inputs['input_ids']
    prompt_len = prompt_ids.shape[1]

    # ---- Stage 1: prompt forward pass + first-token scoring of all catalog items ----
    with torch.no_grad():
        prompt_out = model(input_ids=prompt_ids.to(model.device), use_cache=True)
    last_logit = prompt_out.logits[0, -1, :]
    log_probs_first = F.log_softmax(last_logit, dim=-1)

    stage1_scores = log_probs_first[first_tokens]
    stage1_scores = stage1_scores.masked_fill(~valid_mask, -float('inf'))

    top_n = min(top_n_stage1, int(valid_mask.sum().item()))
    top_n_idx = torch.topk(stage1_scores, top_n).indices.cpu().numpy()

    # ---- Stage 2: full log-likelihood on the survivors, reusing the prompt KV ----
    prompt_kv = prompt_out.past_key_values
    if hasattr(prompt_kv, 'to_legacy_cache'):
        prompt_kv_legacy = prompt_kv.to_legacy_cache()
    else:
        prompt_kv_legacy = prompt_kv

    final_scores = np.full(len(catalog_titles), -float('inf'))

    for idx in top_n_idx:
        title = catalog_titles[idx]
        if not title:
            continue
        full_ids = tokenizer(prompt_text + title, return_tensors='pt')['input_ids']
        if full_ids.shape[1] <= prompt_len:
            continue
        title_ids = full_ids[:, prompt_len:].to(model.device)
        title_len = title_ids.shape[1]

        position_ids = torch.arange(
            prompt_len, prompt_len + title_len, device=model.device
        ).unsqueeze(0)

        with torch.no_grad():
            title_out = model(
                input_ids=title_ids,
                past_key_values=prompt_kv_legacy,
                position_ids=position_ids,
                use_cache=False,
            )

        if title_len == 1:
            stacked = last_logit.unsqueeze(0)
        else:
            stacked = torch.cat(
                [last_logit.unsqueeze(0), title_out.logits[0, :-1, :]], dim=0
            )
        log_probs = F.log_softmax(stacked, dim=-1)
        token_lps = log_probs.gather(1, title_ids[0].unsqueeze(1)).squeeze(1)
        final_scores[idx] = token_lps.mean().item()

    return final_scores

In [ ]:
# Sanity check: ground truth should be in stage-1 top-N and rank highly after stage-2
test_by_user = {ex['user_idx']: ex for ex in genrec_test}
sample = genrec_test[0]
prompt = f"{sample['instruction']}\n\n### input:\n{sample['input']}\n\n### Response:\n"
gt_idx = test_ground_truth[sample['user_idx']]
gt_pos = catalog_item_ids.index(gt_idx)

scores = two_stage_score(prompt, top_n_stage1=TOP_N_STAGE1)
gt_score = scores[gt_pos]
rank = (np.argsort(scores)[::-1] == gt_pos).argmax() + 1

print(f'Ground truth title: "{item_titles[gt_idx]}"')
print(f'  Stage-2 score:      {gt_score:.4f}')
print(f'  Final rank:         {rank} / {len(catalog_titles)}')
print(f'  Survived stage 1:   {gt_score != -float("inf")}')

In [ ]:
# Benchmark: time one full two-stage pass
import time

_ = two_stage_score(prompt, top_n_stage1=10)  # warm up GPU

t0 = time.time()
_ = two_stage_score(prompt, top_n_stage1=TOP_N_STAGE1)
elapsed = time.time() - t0

users = 7288 if N_USERS is None else N_USERS
print(f'1 user, stage1 top-{TOP_N_STAGE1}: {elapsed:.1f}s')
print(f'Projected total for {users} users: ~{elapsed * users / 3600:.1f} hours')

## 4. Run two-stage scoring over all users

In [ ]:
eval_users = list(test_by_user.keys())
if N_USERS is not None:
    eval_users = eval_users[:N_USERS]
print(f'Users to evaluate: {len(eval_users)}')

In [ ]:
CHECKPOINT_PATH = f'{RESULTS_DIR}/genrec_full_catalog_two_stage_checkpoint.pkl'

if os.path.exists(CHECKPOINT_PATH) and not SMOKE_TEST:
    with open(CHECKPOINT_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    full_predictions = ckpt['full_predictions']
    latencies = ckpt['latencies']
    stage1_recall = ckpt['stage1_recall']
    print(f'Resumed from checkpoint: {len(full_predictions)} users done.')
else:
    full_predictions = {}
    latencies = []
    stage1_recall = []  # tracks whether GT survived stage 1 per user (diagnostic)

remaining = [u for u in eval_users if u not in full_predictions]
print(f'Total: {len(eval_users)}, Remaining: {len(remaining)}')

for i, user_idx in enumerate(remaining):
    if user_idx not in test_by_user:
        continue

    example = test_by_user[user_idx]
    prompt = f"{example['instruction']}\n\n### input:\n{example['input']}\n\n### Response:\n"

    t0 = time.time()
    scores = two_stage_score(prompt, top_n_stage1=TOP_N_STAGE1)
    latencies.append(time.time() - t0)

    top_k_idx = np.argsort(scores)[::-1][:TOP_K_TO_SAVE]
    full_predictions[user_idx] = [catalog_item_ids[j] for j in top_k_idx]

    # Diagnostic: did the ground truth survive stage 1?
    gt_idx = test_ground_truth.get(user_idx)
    if gt_idx is not None:
        gt_pos = catalog_item_ids.index(gt_idx)
        stage1_recall.append(scores[gt_pos] != -float('inf'))

    if (i + 1) % CHECKPOINT_EVERY == 0:
        recent = latencies[-CHECKPOINT_EVERY:]
        eta_h = np.mean(recent) * (len(eval_users) - len(full_predictions)) / 3600
        recall = np.mean(stage1_recall) if stage1_recall else 0.0
        print(f'  {len(full_predictions)}/{len(eval_users)} | '
              f'latency: {np.mean(recent):.1f} s/user | '
              f'stage1 GT recall: {recall:.2%} | '
              f'ETA: {eta_h:.1f} h')
        if not SMOKE_TEST:
            with open(CHECKPOINT_PATH, 'wb') as f:
                pickle.dump({
                    'full_predictions': full_predictions,
                    'latencies': latencies,
                    'stage1_recall': stage1_recall,
                }, f)

if not SMOKE_TEST:
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump({
            'full_predictions': full_predictions,
            'latencies': latencies,
            'stage1_recall': stage1_recall,
        }, f)

print(f'\nDone. {len(full_predictions)} users scored.')
if latencies:
    print(f'Mean latency: {np.mean(latencies):.1f} s/user')
if stage1_recall:
    print(f'Stage-1 GT recall: {np.mean(stage1_recall):.2%} ({sum(stage1_recall)}/{len(stage1_recall)})')

## 5. Evaluate

In [ ]:
import math

def hit_at_k(ranked_list, ground_truth, k=10):
    return 1.0 if ground_truth in ranked_list[:k] else 0.0

def ndcg_at_k(ranked_list, ground_truth, k=10):
    for i, item in enumerate(ranked_list[:k]):
        if item == ground_truth:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_ranking(preds, gt, k_values=None):
    if k_values is None:
        k_values = [5, 10, 20]
    results = {}
    for k in k_values:
        hits, ndcgs = [], []
        for uid, true_item in gt.items():
            if uid not in preds:
                continue
            hits.append(hit_at_k(preds[uid], true_item, k))
            ndcgs.append(ndcg_at_k(preds[uid], true_item, k))
        results[f'HR@{k}'] = np.mean(hits) if hits else 0.0
        results[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0.0
    return results


full_ranking = evaluate_ranking(full_predictions, test_ground_truth, k_values=[1, 5, 10, 20])
print(f'Full-catalog ranking — two-stage log-likelihood ({len(full_predictions)} users):')
for m, v in full_ranking.items():
    print(f'  {m}: {v:.4f}')

if stage1_recall:
    print(f'\nStage-1 ground-truth recall: {np.mean(stage1_recall):.2%}')
    print('(if low, increase TOP_N_STAGE1)')

print('\n--- Cross-paradigm full-catalog HR@10 ---')
print(f'  RAG (200 users):                                {0.0500:.4f}')
print(f'  Generative (two-stage LL, this notebook):     {full_ranking["HR@10"]:.4f}')
print(f'  Generative (title gen, fine-tuned):            {0.0266:.4f}')
print(f'  Explainable BPR-MF:                            {0.0220:.4f}')
print(f'  Generative (zero-shot):                        {0.0050:.4f}')

## 6. Save

In [ ]:
if SMOKE_TEST:
    print('*** SMOKE TEST: skipping save ***')
else:
    results = {
        'paradigm': 'generative_finetuned_full_catalog_two_stage',
        'model': f'QLoRA fine-tuned {MODEL_ID} (full-catalog two-stage LL)',
        'anchor_paper': 'GenRec (Ji et al., ECIR 2024)',
        'ranking_full_catalog': full_ranking,
        'system': {
            'latency': {
                'mean_latency_ms': float(np.mean(latencies) * 1000),
                'p50_latency_ms': float(np.median(latencies) * 1000),
                'p95_latency_ms': float(np.percentile(latencies, 95) * 1000),
            },
            'scoring_method': 'two-stage (first-token filter + full LL on survivors)',
            'top_n_stage1': TOP_N_STAGE1,
            'stage1_gt_recall': float(np.mean(stage1_recall)) if stage1_recall else None,
            'catalog_size': len(catalog_titles),
            'top_k_saved': TOP_K_TO_SAVE,
            'n_users_evaluated': len(full_predictions),
        },
    }
    with open(f'{RESULTS_DIR}/generative_full_catalog_two_stage_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    with open(f'{RESULTS_DIR}/generative_full_catalog_two_stage_predictions.pkl', 'wb') as f:
        pickle.dump(full_predictions, f)
    print(f'Results saved to {RESULTS_DIR}/')